# Finetuning Foundation Models

**Finetuning** adapts a pre-trained model to your data before forecasting: the model is trained for a few steps on your time series, which can improve accuracy when your series have a different distribution or scale than the data the model was pre-trained on.

Finetuning in foundationforecast is available for **Chronos 2** and **TimeGPT**. See [ChronosFinetuningConfig](../api/models/foundation/models/#foundationforecast.models.chronos.ChronosFinetuningConfig) and [TimeGPTFinetuningConfig](../api/models/foundation/models/#foundationforecast.models.timegpt.TimeGPTFinetuningConfig) for API references.

## Import libraries

In [ ]:
import os

import pandas as pd
from functools import partial

from foundationforecast import FoundationForecast
from foundationforecast.models.chronos import Chronos, ChronosFinetuningConfig

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mase, mape, scaled_crps


## Load the dataset

We use the same events pageviews dataset as in the [Chronos family](chronos-family.ipynb) example.

In [ ]:
df = pd.read_csv(
    "https://timecopilot.s3.amazonaws.com/public/data/events_pageviews.csv",
    parse_dates=["ds"],
)
df.head()


## Chronos 2

Chronos 2 supports finetuning via **`ChronosFinetuningConfig`** (see the [API reference](../api/models/foundation/models/#foundationforecast.models.chronos.ChronosFinetuningConfig)). You pass an instance to the `Chronos` constructor; when you call `forecast()`, the model is finetuned on the context data before predicting.

**Supported parameters:**

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `finetune_steps` | int | 1000 | Number of training steps. Maps to the chronos pipeline's `num_steps`. |
| `learning_rate` | float or None | None → 1e-6 | Optimizer learning rate (chronos uses 1e-6; for LoRA, 1e-5 is recommended). |
| `batch_size` | int or None | None → 256 | Training batch size for finetuning. The `batch_size` on `Chronos` is for inference only. |
| `finetune_mode` | "full" or "lora" or None | None → "full" | Full parameter update vs. LoRA. |
| `lora_config` | object or None | None | LoRA configuration when `finetune_mode="lora"`; see the [Chronos-2 quickstart](https://github.com/amazon-science/chronos-forecasting/blob/main/notebooks/chronos-2-quickstart.ipynb) for details. |
| `save_path` | str or Path or None | None | If set, the finetuned model is saved to this directory. Use the same path as `repo_id` with `finetuning_config=None` to load and reuse it for later forecasts. |

The forecast horizon `h` you pass to `forecast(df, h, ...)` is used as `prediction_length` for the internal `fit()` call.

### Compare Chronos 2 with and without finetuning

In [ ]:
# Chronos 2 without finetuning
chronos2 = Chronos(repo_id="autogluon/chronos-2-small", alias="chronos2")
# Chronos 2 with finetuning (fewer steps for a quicker example)
chronos2_finetuned = Chronos(
    repo_id="autogluon/chronos-2-small",
    alias="chronos-2-finetuned",
    finetuning_config=ChronosFinetuningConfig(
        finetune_steps=10,
        save_path="./chronos-2-finetuned-path/", # optional, save the finetuned model
    ),
)
# Chronos 2 with LoRA finetuning
chronos2_lora = Chronos(
    repo_id="autogluon/chronos-2-small",
    alias="chronos-2-finetuned-lora",
    finetuning_config=ChronosFinetuningConfig(
        finetune_mode="lora",
        finetune_steps=10,
        learning_rate=1e-5,
        save_path="./chronos-2-finetuned-lora-path/", # optional, save the finetuned model
    ),
)
auto_arima =models = [chronos2, chronos2_finetuned, chronos2_lora, auto_arima]
ff = FoundationForecast(models=models)
level = [20, 40, 60, 80]
cv_df = ff.cross_validation(df=df, h=12, level=level)
cv_df.head()


In [ ]:
ff.plot(df, cv_df.drop(columns=["cutoff", "y"]), level=[80])


In [ ]:
eval_compare = evaluate(
    cv_df.drop(columns=["cutoff"]),
    train_df=df.query("ds <= '2024-08-31'"),
    metrics=[partial(mase, seasonality=12), scaled_crps],
    level=level,
)
eval_compare.groupby("metric").mean(numeric_only=True).T.sort_values(by="scaled_crps").round(3)


## Using the best model for forecasts

The evaluation showed that **Chronos 2 finetuned with LoRA** had the best (lowest) scaled CRPS among the compared models. Because we saved it with `save_path="./chronos-2-finetuned-lora/"`, we can load it and use it for new forecasts without finetuning again.

In [ ]:
best_model_path = "chronos-2-finetuned-lora-path"
best_model = Chronos(
    repo_id=best_model_path,
    finetuning_config=None,
    batch_size=2,
    alias="chronos2-best",
)
fcst_best = best_model.forecast(df, h=12, freq="MS", level=[80, 90])
fcst_best.head()

The forecasts above come from the best-performing model in our comparison (Chronos 2 with LoRA finetuning), loaded from the saved checkpoint. You can reuse this path in other notebooks or scripts for inference without running finetuning again.

In [ ]:
best_model.plot(df, fcst_best, level=[80, 90])

## Finetuning evaluation

We compare **MAPE** across different `finetune_steps` (and a baseline with no finetuning) by running cross-validation and evaluating with MAPE. You can use the results to choose a finetuning step for production. In this example the target `y` is never zero in the evaluation period, so MAPE is well-defined; in general you should be careful with MAPE (it is undefined or unstable when `y` is zero or very small). We use it here for illustration only.

**Note:** This experiment finetunes the model **on the fly**. In every cross-validation window, the model is finetuned on the data up to the cutoff, then forecasts the next `h` steps. No pre-saved finetuned checkpoint is reused between windows, each window gets a fresh finetuning run.

**Parameters**

| Parameter | Example value | Description |
|-----------|----------------|-------------|
| `h` | 24 | Forecast horizon (number of steps to predict per window). |
| `n_windows` | 4 | Number of cross-validation windows (backtest pivots). |
| `finetune_steps` (candidates) | `range(10, 50, 10)` | Finetuning steps to compare; include baseline (no finetuning) to compare against. |
| `repo_id` | autogluon/chronos-2-small | Chronos model to use (and optionally finetune). |
| Metric | MAPE | Metric used to compare configs (here MAPE for illustration; choose one appropriate for your use case). |

In [ ]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/refs/heads/main/ETT-small/ETTh1.csv",
    parse_dates=["date"],
).melt("date", var_name="unique_id", value_name="y")
df.rename(columns={"date": "ds"}, inplace=True)
df = df.groupby("unique_id").tail(24 * 7 * 7)

In [ ]:
df.groupby("unique_id").tail(4 * 24).query("y==0")

In [ ]:
FoundationForecast.plot(df)

In [ ]:
h = 24
n_windows = 4

# Baseline (no finetuning) + 4 finetune step values — one forecaster per config
finetune_steps_list = range(10, 50, 10)
models_eval = [
    Chronos(repo_id="autogluon/chronos-2-small", alias="Chronos2-baseline"),
] + [
    Chronos(
        repo_id="autogluon/chronos-2-small",
        alias=f"Chronos2-steps{s}",
        finetuning_config=ChronosFinetuningConfig(
            finetune_steps=s,
            finetune_mode="lora",
            learning_rate=1e-5,
        ),
    )
    for s in finetune_steps_list
]

ff_eval = FoundationForecast(models=models_eval)
cv_df_eval = ff_eval.cross_validation(df=df, h=h, n_windows=n_windows)

eval_eval = evaluate(
    cv_df_eval.drop(columns=["cutoff"]),
    train_df=df,
    metrics=[mape],
)
model_cols = ["Chronos2-baseline"] + [f"Chronos2-steps{s}" for s in finetune_steps_list]
avg_mape = eval_eval[eval_eval["metric"] == "mape"][model_cols].mean()
avg_mape.rename("mape").sort_values().round(4)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Build a tidy table with baseline + finetuned variants
mape_table = pd.DataFrame(
    {
        "model": model_cols,
        "mape": [float(avg_mape[m]) for m in model_cols],
    }
)
mape_table["finetune_steps"] = mape_table["model"].map(
    lambda m: 0 if m == "Chronos2-baseline" else int(m.replace("Chronos2-steps", ""))
)
mape_table = mape_table.sort_values("finetune_steps")

# Plot MAPE vs finetuning steps (0 = baseline, no finetuning)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(
    mape_table["finetune_steps"],
    mape_table["mape"],
    marker="o",
    linewidth=2,
)
ax.set_xlabel("finetune_steps (0 = baseline)")
ax.set_ylabel("MAPE")
ax.set_title("MAPE vs finetune_steps")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

mape_table

### Conclusion

Use the plot (and table) to pick the `finetune_steps` value with the **lowest MAPE**; that is the best candidate for this dataset under this backtest setup. In this notebook, `finetune_steps=0` represents the baseline (no finetuning), so you can directly verify whether finetuning improves over baseline.

Because finetuning here runs **on the fly in each cross-validation window**, this result reflects the cost/benefit of repeatedly finetuning before each forecast window.

## TimeGPT

TimeGPT supports finetuning via **`TimeGPTFinetuningConfig`** (see the [API reference](../api/models/foundation/models/#foundationforecast.models.timegpt.TimeGPTFinetuningConfig)). You pass an instance to the `TimeGPT` constructor; when you call `forecast()`, the model is finetuned on the context data before predicting.

> **Note:** TimeGPT requires a valid API key from Nixtla. Set the `NIXTLA_API_KEY` environment variable or pass `api_key` to the constructor. You can get a key at [dashboard.nixtla.io](https://dashboard.nixtla.io).

**Supported parameters:**

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `finetune_steps` | int | 10 | Number of training iterations to minimize forecasting error. |
| `finetune_loss` | `"default"`, `"mae"`, `"mse"`, `"rmse"`, `"mape"`, or `"smape"` | `"default"` | Loss function used during finetuning. |
| `finetune_depth` | 1–5 | 1 | How many model layers to finetune (1 = few, 5 = entire model). |

In [ ]:
from foundationforecast.models.timegpt import TimeGPT, TimeGPTFinetuningConfig

In [ ]:
# TimeGPT without finetuning
timegpt = TimeGPT(alias="TimeGPT")
# TimeGPT with finetuning
timegpt_finetuned = TimeGPT(
    finetuning_config=TimeGPTFinetuningConfig(
        finetune_steps=10,
        finetune_loss="mse",
    ),
    alias="TimeGPT-finetuned",
)

### Forecast with finetuned Chronos and TimeGPT together

Both finetuned models can be used in a single `FoundationForecast` call:

In [ ]:
models = [chronos2_lora, timegpt, timegpt_finetuned]
ff = FoundationForecast(models=models)
fcst_df = ff.forecast(df=df, h=24)
fcst_df.head()